In [ ]:
# --- Cell 1: setup & paths ---
from google.colab import drive
drive.mount('/content/drive')

import os, platform
import pandas as pd

PROJECT = "/content/drive/MyDrive/dlbcl-prognosis"
RAW = os.path.join(PROJECT, "data", "raw")
os.makedirs(RAW, exist_ok=True)

DATASETS = ["GSE10846", "GSE31312", "GSE117556"]
print("Python", platform.python_version(), "| pandas", pd.__version__)
print("Raw data dir:", RAW, "| exists:", os.path.isdir(RAW))

In [ ]:
# --- Cell 2: locate series-matrix files and stream only their metadata header ---
import csv, gzip, re, urllib.request

def geo_matrix_urls(gse):
    """Return URLs of all series-matrix files for a GEO series."""
    base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{gse[:-3]}nnn/{gse}/matrix/"
    html = urllib.request.urlopen(base, timeout=60).read().decode()
    files = sorted(set(re.findall(r'href="([^"]*_series_matrix\.txt\.gz)"', html)))
    return [base + f for f in files]

def remote_size_mb(url):
    """Size of the full compressed file, without downloading it."""
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req, timeout=60) as r:
        return int(r.headers.get("Content-Length", 0)) / 1e6

def stream_header(url):
    """Read only the metadata block; stop before the expression table starts."""
    lines = []
    with urllib.request.urlopen(url, timeout=300) as r, gzip.GzipFile(fileobj=r) as gz:
        for raw in gz:
            line = raw.decode("utf-8", errors="replace").rstrip("\r\n")
            if line.startswith("!series_matrix_table_begin"):
                break
            lines.append(line)
    return lines

headers = {}
for gse in DATASETS:
    urls = geo_matrix_urls(gse)
    print(f"\n{gse}: {len(urls)} matrix file(s)")
    headers[gse] = []
    for url in urls:
        fname = url.rsplit("/", 1)[-1]
        size = remote_size_mb(url)
        lines = stream_header(url)
        with open(os.path.join(RAW, fname.replace(".txt.gz", "_header.txt")), "w") as f:
            f.write("\n".join(lines))
        headers[gse].append(lines)
        print(f"  {fname} | full size ~ {size:.0f} MB | header lines: {len(lines)}")

In [ ]:
# --- Cell 3: parse sample metadata into one row per sample and summarize ---
def split_kv(v):
    if ": " in v:
        return v.rsplit(": ", 1)
    if ":" in v:
        return v.rsplit(":", 1)
    return "unlabeled", v

def parse_header(lines):
    rows = [next(csv.reader([l], delimiter="\t")) for l in lines if l.startswith("!Sample_")]
    n = len(rows[0]) - 1
    recs = [{} for _ in range(n)]
    keep = {"!Sample_geo_accession": "gsm", "!Sample_title": "title",
            "!Sample_platform_id": "platform", "!Sample_source_name_ch1": "source"}
    for row in rows:
        key, vals = row[0], row[1:]
        if key in keep:
            for i, v in enumerate(vals):
                recs[i][keep[key]] = v
        elif key == "!Sample_characteristics_ch1":
            for i, v in enumerate(vals):
                v = v.strip()
                if not v:
                    continue
                k, val = split_kv(v)
                k = k.strip()
                while k in recs[i]:          # never silently overwrite a duplicate key
                    k += "_dup"
                recs[i][k] = val.strip()
    return pd.DataFrame(recs)

def summarize(df, max_vals=4, width=45):
    for col in df.columns:
        if col in ("gsm", "title"):
            continue
        s = df[col]
        top = "; ".join(f"{str(k)[:width]} ({v})" for k, v in s.value_counts().head(max_vals).items())
        print(f"  - {col[:60]} | missing={s.isna().sum()} | unique={s.nunique()} | {top}")

clinical = {}
for gse in DATASETS:
    df = pd.concat([parse_header(l) for l in headers[gse]], ignore_index=True)
    clinical[gse] = df
    df.to_csv(os.path.join(RAW, f"{gse}_clinical_raw.csv"), index=False)
    print(f"\n===== {gse}: {len(df)} samples | platforms: {df['platform'].value_counts().to_dict()} =====")
    summarize(df)

In [ ]:
# --- Cell 4: rough keyword-based inventory of what each dataset contains ---
patterns = {
    "survival_time":   r"follow|time|year|month|day|\bos\b|pfs|efs|surviv",
    "survival_status": r"status|event|dead|death|alive|censor",
    "cell_of_origin":  r"abc|gcb|origin|coo|subtype|diagnosis|classif",
    "ipi_clinical":    r"ipi|ldh|stage|ecog|extranodal|age",
    "treatment":       r"chemo|treat|regimen|arm|chop|rituximab",
}
rows = []
for gse, df in clinical.items():
    row = {"dataset": gse, "n": len(df)}
    for tag, pat in patterns.items():
        hits = [c for c in df.columns if re.search(pat, c, flags=re.I)]
        row[tag] = ", ".join(hits) if hits else "-"
    rows.append(row)

pd.set_option("display.max_colwidth", 80)
inventory = pd.DataFrame(rows).set_index("dataset")
inventory.to_csv(os.path.join(RAW, "dataset_inventory.csv"))
inventory.T